# Homework April 28th, 2026
#### Task 1: Find 2-3 users with at least 10 events and map their journey. 
You can use BigQuery directly. I also encourage you to fetch the data via the BigQuery API using Python and do the analysis there.
Here, you will need to master the UNNEST function.

#### Task 2: Plan how you will use this data to create a high-level Macro View funnel analysis (plan only required)

#### Task 3: Research Tableau Public or other BI dashboards for funnel analysis dashboards. Read articles on what is and isn't best practices. Show me an example and explain it to me as if you built it yourself.

## Data Import and Validation

Before doing the tasks, first need to connect to the data source (Google Cloud) and validate access as per below code.

#### Needed libraries
- `google-cloud-bigquery` — BigQuery client
- `pandas` — tabular results
- `db-dtypes` — BigQuery types (e.g. `DATE`, `TIME`) in DataFrames
- `pyarrow` — efficient columnar transfer



In [ ]:
# Install helper tools (Python libraries) this notebook needs.
# You only need to run this cell once per computer / environment; it downloads quietly (-q).
%pip install -q google-cloud-bigquery pandas db-dtypes pyarrow

In [2]:
# -----------------------------------------------------------------------------
# Connect to Google BigQuery (Google's database service in the cloud).
# Think of "client" as the door you open to ask questions and get tables back.
# -----------------------------------------------------------------------------
import os
from pathlib import Path

from google.cloud import bigquery

# Your Google Cloud *project* id — this is who pays for queries and holds your permissions.
BILLING_PROJECT_ID = "project-92e3923f-0e27-4c05-b11"

# Login file: a service-account JSON key from Google Cloud (keep it private).
# If this file sits in the same folder you opened Jupyter from, we pick it up automatically.
_credentials_path = Path.cwd() / "google_creds.json"
if _credentials_path.is_file():
    # Tell Google libraries where to find the key (only set if not already set elsewhere).
    os.environ.setdefault("GOOGLE_APPLICATION_CREDENTIALS", str(_credentials_path.resolve()))

# Create the live connection object used by later cells.
client = bigquery.Client(project=BILLING_PROJECT_ID)

# Tiny test question: proves the connection works before we touch any analytics data.
_smoke = client.query("SELECT 1 AS ok").to_dataframe()
assert _smoke["ok"].iloc[0] == 1
print("BigQuery client ready (project:", BILLING_PROJECT_ID + ")")

/opt/anaconda3/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


BigQuery client ready (project: project-92e3923f-0e27-4c05-b11)


## Task 1 — User journeys (≥10 events, `UNNEST`)

**Goal:** pick **2–3** `user_pseudo_id` values with at least **10** events, pull their full timelines, and enrich each event using **`UNNEST(event_params)`** (and **`UNNEST(items)`** where present) so the journey is readable (page / title / session id, plus product hints).

**Why `UNNEST`:** GA4 stores repeated keys in `event_params` and line items in `items`; you flatten or filter those arrays in SQL, then analyze in Python.

**Scope:** the snippet below scans `events_*` for **January 2021** only (adjust `_TABLE_SUFFIX` bounds if you need more users). Tight bounds keep bytes scanned predictable.

In [ ]:
# =============================================================================
# USER JOURNEYS (plain-language overview)
# =============================================================================
# We use Google's *sample* shop data (not your own website). Each row is one
# "event" (e.g. viewed a page, added to cart). Many details sit inside nested
# lists — SQL's UNNEST() is like "open the list and read line by line."
#
# Flow: (1) load events for a date range → (2) count events per shopper id →
# (3) keep the busiest shoppers → (4) pull their full story with readable
# page titles / URLs / product names → (5) print timelines in Python.
# =============================================================================

import pandas as pd  # Used here to check "is this session id missing?" safely.

# Only keep anonymous shopper ids with at least this many events (homework asked for ≥10).
MIN_EVENTS = 10
# How many of those "heavy" shoppers we will show in full (homework asked for 2–3).
NUM_USERS = 3

# Public GA4 tables: one table per calendar day; * matches many days at once.
JOURNEY_DATASET = "`bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`"
# Limit which daily tables we read (YYYYMMDD). Smaller range = faster / cheaper.
_TABLE_SUFFIX_START = "20210101"
_TABLE_SUFFIX_END = "20210131"

journey_sql = f"""
-- Step A: raw events (one row = one thing that happened on the site).
WITH raw AS (
  SELECT
    user_pseudo_id,   -- Google's anonymous id for that browser/app (not a human name).
    event_timestamp,  -- When it happened (stored in tiny time units; we convert later).
    event_name,       -- Short label, e.g. page_view, purchase.
    event_date,       -- Calendar day as text YYYYMMDD.
    event_params,     -- Extra details as a *list* of key/value pairs (needs UNNEST to read).
    items             -- Products attached to the event, also a *list* (needs UNNEST).
  FROM {JOURNEY_DATASET}
  WHERE _TABLE_SUFFIX BETWEEN '{_TABLE_SUFFIX_START}' AND '{_TABLE_SUFFIX_END}'
),
-- Step B: count how busy each shopper is; drop anyone with too few events.
user_event_counts AS (
  SELECT user_pseudo_id, COUNT(*) AS n_events
  FROM raw
  GROUP BY user_pseudo_id
  HAVING n_events >= {MIN_EVENTS}
),
-- Step C: take the top few busiest shoppers (by event count).
top_users AS (
  SELECT user_pseudo_id
  FROM user_event_counts
  ORDER BY n_events DESC
  LIMIT {NUM_USERS}
),
-- Step D: keep only events belonging to those shoppers.
events_for_users AS (
  SELECT r.*
  FROM raw r
  INNER JOIN top_users t USING (user_pseudo_id)
),
-- Step E: one row per event again, but with human-friendly columns pulled out of lists.
-- UNNEST opens the list; we pick the row whose "key" matches what we want (page URL, etc.).
enriched AS (
  SELECT
    e.user_pseudo_id,
    e.event_date,
    TIMESTAMP_MICROS(e.event_timestamp) AS event_ts,  -- Turn micro-timestamps into normal date-times.
    e.event_name,
    (SELECT ep.value.string_value FROM UNNEST(e.event_params) AS ep WHERE ep.key = 'page_location' LIMIT 1)
      AS page_location,
    (SELECT ep.value.string_value FROM UNNEST(e.event_params) AS ep WHERE ep.key = 'page_title' LIMIT 1)
      AS page_title,
    (SELECT ep.value.int_value FROM UNNEST(e.event_params) AS ep WHERE ep.key = 'ga_session_id' LIMIT 1)
      AS ga_session_id,
    (SELECT COALESCE(ep.value.string_value, CAST(ep.value.int_value AS STRING)) FROM UNNEST(e.event_params) AS ep WHERE ep.key = 'session_engaged' LIMIT 1)
      AS session_engaged,
    -- Product names on this event (joined with " | " so one cell can show several items).
    (
      SELECT STRING_AGG(SUBSTR(COALESCE(it.item_name, ''), 1, 80), ' | ' ORDER BY ord)
      FROM UNNEST(e.items) AS it WITH OFFSET AS ord
    ) AS item_names_agg
  FROM events_for_users AS e
)
SELECT * FROM enriched
ORDER BY user_pseudo_id, event_ts
"""

# Send the big question to Google; results come back as a familiar spreadsheet-style table.
journey_df = client.query(journey_sql).to_dataframe()

# Quick headline numbers before the long printout.
print("Selected users:", journey_df["user_pseudo_id"].drop_duplicates().tolist())
print("Total pulled rows:", len(journey_df))

# -----------------------------------------------------------------------------
# Pretty print: walk each shopper's rows from oldest → newest (their "story").
# -----------------------------------------------------------------------------
for uid, grp in journey_df.groupby("user_pseudo_id", sort=False):
    g = grp.sort_values("event_ts").reset_index(drop=True)
    print("\n" + "=" * 88)
    print(f"user_pseudo_id={uid}  |  events={len(g)}")
    print("=" * 88)
    for i, row in g.iterrows():
        step = i + 1  # Step number in that person's journey (1, 2, 3, ...).
        loc = (row["page_location"] or "")[:72]  # Shorten very long URLs for the screen.
        title = (row["page_title"] or "")[:48]
        items = row["item_names_agg"] or ""
        if items:
            items = items[:100] + ("…" if len(str(row["item_names_agg"])) > 100 else "")
        sid = row["ga_session_id"]
        sid_str = f"session={sid}" if pd.notna(sid) else "session=?"
        print(
            f"{step:3d}  {row['event_ts']}  {row['event_name']!s:22s}  {sid_str}  |  {title}"
        )
        if loc:
            print(f"      ↳ {loc}")
        if items:
            print(f"      ↳ items: {items}")

# First 20 rows of the table (useful if you prefer scrolling to the long text above).
display(journey_df.head(20))

/opt/anaconda3/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Selected users: ['2522139.8383510968', '27488559.4267170544', '72522849.6381089738']
Total pulled rows: 3006

user_pseudo_id=2522139.8383510968  |  events=1083
  1  2021-01-20 17:17:23.697043+00:00  session_start           session=5209072058  |  Home
      ↳ https://shop.googlemerchandisestore.com/
  2  2021-01-20 17:17:23.697043+00:00  page_view               session=5209072058  |  Home
      ↳ https://shop.googlemerchandisestore.com/
  3  2021-01-20 17:17:23.697043+00:00  first_visit             session=5209072058  |  Home
      ↳ https://shop.googlemerchandisestore.com/
  4  2021-01-20 17:17:28.711948+00:00  page_view               session=5209072058  |  Home
      ↳ https://shop.googlemerchandisestore.com/
  5  2021-01-20 17:17:28.711948+00:00  view_promotion          session=5209072058  |  Home
      ↳ https://shop.googlemerchandisestore.com/
      ↳ items: (not set)
  6  2021-01-20 17:17:32.152294+00:00  user_engagement         session=5209072058  |  Home
      ↳ https://shop.goo

,user_pseudo_id,event_date,event_ts,event_name,page_location,page_title,ga_session_id,session_engaged,item_names_agg
0,2522139.8383510968,20210120,2021-01-20 17:17:23.697043+00:00,session_start,https://shop.googlemerchandisestore.com/,Home,5209072058,None,None
1,2522139.8383510968,20210120,2021-01-20 17:17:23.697043+00:00,page_view,https://shop.googlemerchandisestore.com/,Home,5209072058,0,None
2,2522139.8383510968,20210120,2021-01-20 17:17:23.697043+00:00,first_visit,https://shop.googlemerchandisestore.com/,Home,5209072058,1,None
3,2522139.8383510968,20210120,2021-01-20 17:17:28.711948+00:00,page_view,https://shop.googlemerchandisestore.com/,Home,5209072058,1,None
4,2522139.8383510968,20210120,2021-01-20 17:17:28.711948+00:00,view_promotion,https://shop.googlemerchandisestore.com/,Home,5209072058,1,(not set)
5,2522139.8383510968,20210120,2021-01-20 17:17:32.152294+00:00,user_engagement,https://shop.googlemerchandisestore.com/,Home,5209072058,1,None
6,2522139.8383510968,20210120,2021-01-20 17:17:37.577488+00:00,page_view,https://shop.googlemerchandisestore.com/Google...,Sale | Google Merchandise Store,5209072058,1,None
7,2522139.8383510968,20210120,2021-01-20 17:18:04.085440+00:00,scroll,https://shop.googlemerchandisestore.com/Google...,Sale | Google Merchandise Store,5209072058,1,None
8,2522139.8383510968,20210120,2021-01-20 17:20:27.552119+00:00,user_engagement,https://shop.googlemerchandisestore.com/Google...,Sale | Google Merchandise Store,5209072058,1,None
9,2522139.8383510968,20210120,2021-01-20 17:20:32.953383+00:00,scroll,https://shop.googlemerchandisestore.com/Google...,Sale | Google Merchandise Store,5209072058,1,None
